YouTube Intelligence Assistant with Local LLM Tool Calling

Build a local AI assistant that can understand and analyze YouTube content by giving an LLM the ability to interact with YouTube's ecosystem through custom-built tools. Using Ollama to run a language model entirely on machine, design specialized functions that extract video transcripts, fetch metadata, search for content, and retrieve trending videos — then wire them together so the model can call them on demand. This project teaches complete manual tool calling cycle: from how the model decides which tool to use, to executing myself and feeding results back into the conversation. 

In [2]:
pip install ollama youtube-transcript-api yt-dlp

   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   --------- ------------------------------ 0.8/3.3 MB 4.2 MB/s eta 0:00:01
   ------------------------- -------------- 2.1/3.3 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 3.3/3.3 MB 5.7 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [16]:
import ollama

# Extract and print only the model names
for m in ollama.list().models:
    print(m.model)


llava:latest
llama3.2:latest
qwen2.5:3b
qwen3.5:2b
gemma3:270m


In [15]:
import requests
print(requests.get('http://localhost:11434').text)


Ollama is running


In [19]:
import sys
print(sys.version)


3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [22]:
def extract_video_id(url: str) -> str:
    """Extract the video ID from a YouTube URL.
    
    Args:
        url: The full YouTube video URL
        
    Returns:
        The video ID string extracted from the URL
    """
    pattern = r"(?:https?://)?(?:www\.)?(?:youtube\.com/watch\?v=|youtu\.be/)([0-9A-Za-z_-]{11})"
    match = re.search(pattern, url)
    if match:
        return match.group(1)
    return "Could not extract video ID from the given URL"

How It Works
Input: It accepts a full website link (URL) as a text string.
Pattern Matching: It uses a Regular Expression (re) to search for common YouTube link structures.
Extraction: It looks for either ://youtube.com or the shortened youtu.be/ format
Output: It isolates and returns the 11 characters representing the specific video ID.
Fallback: It returns an error message if the link format is invalid or unsupported.

In [24]:

print(extract_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ"))
print(extract_video_id("https://youtu.be/dQw4w9WgXcQ"))
print(extract_video_id("https://not-a-youtube-url.com"))

dQw4w9WgXcQ
dQw4w9WgXcQ
Could not extract video ID from the given URL


In [27]:
from youtube_transcript_api import YouTubeTranscriptApi

def get_transcript(video_id: str) -> str:
    """Fetch the full transcript of a YouTube video.
    
    Args:
        video_id: The YouTube video ID
        
    Returns:
        The full transcript of the video as a single string
    """
    try:
        ytt_api = YouTubeTranscriptApi()
        fetched_transcript = ytt_api.fetch(video_id)
        transcript = " ".join([snippet.text for snippet in fetched_transcript])
        return transcript[:3000]
    except Exception as e:
        return f"Could not fetch transcript: {str(e)}"

This Python code utilizes the youtube_transcript_api library to fetch and consolidate the text from a YouTube video's closed captions. It merges individual caption segments into a single string and truncates the output to 3,000 characters to manage data volume.

In [28]:
#Quick test 
video_id = extract_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ")
print(get_transcript(video_id))

[♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules
and so do I ♪ ♪ A full commitment's
what I'm thinking of ♪ ♪ You wouldn't get this
from any other guy ♪ ♪ I just wanna tell you
how I'm feeling ♪ ♪ Gotta make you understand ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ We've known each other
for so long ♪ ♪ Your heart's been aching
but you're too shy to say it ♪ ♪ Inside we both know
what's been going ♪ ♪ We know the game
and we're gonna play it ♪ ♪ And if you ask me
how I'm feeling ♪ ♪ Don't tell me
you're too blind to see ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie
and hurt you ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around
and desert you ♪ ♪ Never gon

Tool 3 — Get Video Metadata

In [31]:
import yt_dlp

def get_video_metadata(video_id: str) -> dict:
    """Fetch metadata for a YouTube video including title, views, duration and author.
    
    Args:
        video_id: The YouTube video ID
        
    Returns:
        A dictionary containing the video metadata
    """
    try:
        url = f"https://www.youtube.com/watch?v={video_id}"
        ydl_opts = {
            "quiet": True,
            "skip_download": True,
            "no_warnings": True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
            return {
                "title": info.get("title"),
                "author": info.get("uploader"),
                "views": info.get("view_count"),
                "duration_seconds": info.get("duration"),
                "description": info.get("description", "")[:500],
            }
    except Exception as e:
        return {"error": f"Could not fetch metadata: {str(e)}"}

In [32]:
# Quick test
video_id = extract_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ")
print(get_video_metadata(video_id))

{'title': 'Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)', 'author': 'Rick Astley', 'views': 1776619567, 'duration_seconds': 213, 'description': 'The official video for “Never Gonna Give You Up” by Rick Astley. \n\nNever: The Autobiography 📚 OUT NOW! \nFollow this link to get your copy and listen to Rick’s ‘Never’ playlist ❤️ #RickAstleyNever\nhttps://linktr.ee/rickastleynever\n\n“Never Gonna Give You Up” was a global smash on its release in July 1987, topping the charts in 25 countries including Rick’s native UK and the US Billboard Hot 100.  It also won the Brit Award for Best single in 1988. Stock Aitken and Waterman wrote and produced the t'}


Tool 4 — Search YouTube

In [33]:
def search_youtube(query: str, max_results: int = 5) -> list:
    """Search YouTube for videos matching a query.
    
    Args:
        query: The search query string
        max_results: Maximum number of results to return, defaults to 5
        
    Returns:
        A list of dictionaries containing video title, url and video_id
    """
    try:
        ydl_opts = {
            "quiet": True,
            "no_warnings": True,
            "skip_download": True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            results = ydl.extract_info(f"ytsearch{max_results}:{query}", download=False)
            videos = []
            for entry in results["entries"]:
                videos.append({
                    "title": entry.get("title"),
                    "video_id": entry.get("id"),
                    "url": f"https://www.youtube.com/watch?v={entry.get('id')}",
                    "views": entry.get("view_count"),
                    "duration_seconds": entry.get("duration"),
                })
            return videos
    except Exception as e:
        return [{"error": f"Could not search YouTube: {str(e)}"}]

In [34]:
# Quick test
results = search_youtube("python tutorials", max_results=3)
for video in results:
    print(video)

{'title': 'Python Tutorial - Python Full Course for Beginners in Tamil', 'video_id': 'm67-bOpOoPU', 'url': 'https://www.youtube.com/watch?v=m67-bOpOoPU', 'views': 7523011, 'duration_seconds': 34684}
{'title': 'Python Full Course for Beginners', 'video_id': 'K5KVEU3aaeQ', 'url': 'https://www.youtube.com/watch?v=K5KVEU3aaeQ', 'views': 6387687, 'duration_seconds': 7340}
{'title': 'Python Full Course for Beginners', 'video_id': '_uQrJ0TkZlc', 'url': 'https://www.youtube.com/watch?v=_uQrJ0TkZlc', 'views': 47803286, 'duration_seconds': 22447}


Tool 5 — Get Trending Videos

In [37]:
def get_trending_videos(max_results: int = 5) -> list:
    """Fetch currently trending videos on YouTube by searching for popular content today.
    
    Args:
        max_results: Maximum number of trending videos to return, defaults to 5
        
    Returns:
        A list of dictionaries containing trending video title, url and video_id
    """
    try:
        ydl_opts = {
            "quiet": True,
            "no_warnings": True,
            "skip_download": True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            results = ydl.extract_info(
                "ytsearch10:trending today", download=False
            )
            videos = []
            for entry in results["entries"][:max_results]:
                videos.append({
                    "title": entry.get("title"),
                    "video_id": entry.get("id"),
                    "url": f"https://www.youtube.com/watch?v={entry.get('id')}",
                    "views": entry.get("view_count"),
                })
            return videos
    except Exception as e:
        return [{"error": f"Could not fetch trending videos: {str(e)}"}]

In [38]:
# Quick test
trending = get_trending_videos(max_results=3)
for video in trending:
    print(video)

{'title': 'ASMR Live 🔴 #live #ASMR #satisfying #shorts #shortslive #shortsfeed #trending #coin 2026-05-27 12:16', 'video_id': 'KBGb55ZqMmY', 'url': 'https://www.youtube.com/watch?v=KBGb55ZqMmY', 'views': 4061}
{'title': 'Test Your Brain⚡ #YouTubeLive #LiveNow #Trending #viral 2026-05-27 12:16', 'video_id': 'PbjcOBB2jOE', 'url': 'https://www.youtube.com/watch?v=PbjcOBB2jOE', 'views': 5346}
{'title': 'Ak 111k Live #ak111k #freefire #gamer #trending #ff #youtube #game #viral #pcgaming #pcgaming #trend 2026-05-27 12:16', 'video_id': 'A3OGF8AgSZs', 'url': 'https://www.youtube.com/watch?v=A3OGF8AgSZs', 'views': 15}


Connecting Tools to Ollama

In [39]:
import ollama

# Tool registry — maps tool names to actual functions
tool_registry = {
    "extract_video_id": extract_video_id,
    "get_transcript": get_transcript,
    "get_video_metadata": get_video_metadata,
    "search_youtube": search_youtube,
    "get_trending_videos": get_trending_videos,
}

In [41]:
# Step 1 — Send user message to Ollama with tools available
user_message = "Search for videos about machine learning"

response = ollama.chat(
    model="qwen2.5:3b",
    messages=[{"role": "user", "content": user_message}],
    tools=list(tool_registry.values()),
)

print("Model response:")
print(response.message)

# Step 2 — Extract the tool call from the response
tool_call = response.message.tool_calls[0]
function_name = tool_call.function.name
function_args = tool_call.function.arguments

print(f"\nTool requested: {function_name}")
print(f"Arguments: {function_args}")

# Step 3 — Look up the function in the registry and execute it
function_to_call = tool_registry[function_name]
tool_result = function_to_call(**function_args)

print(f"\nTool result: {tool_result}")

Model response:
role='assistant' content='' thinking=None images=None tool_name=None tool_calls=[ToolCall(function=Function(name='search_youtube', arguments={'query': 'machine learning', 'max_results': 5}))]

Tool requested: search_youtube
Arguments: {'query': 'machine learning', 'max_results': 5}

Tool result: [{'title': 'Complete Machine Learning In 6 Hours| Krish Naik', 'video_id': 'JxgmHe2NyeY', 'url': 'https://www.youtube.com/watch?v=JxgmHe2NyeY', 'views': 1950362, 'duration_seconds': 23872}, {'title': 'Machine Learning | What Is Machine Learning? | Introduction To Machine Learning | 2026 | Simplilearn', 'video_id': 'ukzFI9rgwfU', 'url': 'https://www.youtube.com/watch?v=ukzFI9rgwfU', 'views': 5443196, 'duration_seconds': 472}, {'title': 'AI, Machine Learning, Deep Learning and Generative AI Explained', 'video_id': 'qYNweeDHiyU', 'url': 'https://www.youtube.com/watch?v=qYNweeDHiyU', 'views': 3165839, 'duration_seconds': 600}, {'title': "Most Machine Learning Courses Won't Get You H

Send Results Back to Ollama

In [42]:
# Step 4 — Send tool result back to the model
messages = [
    {"role": "user", "content": user_message},
    response.message,
    {"role": "tool", "tool_name": function_name, "content": str(tool_result)},
]

final_response = ollama.chat(
    model="qwen2.5:3b",
    messages=messages,
    tools=list(tool_registry.values()),
)

print("\nFinal answer:")
print(final_response.message.content)


Final answer:
Here are some videos about machine learning that you might find interesting:

1. **Title**: Complete Machine Learning In 6 Hours| Krish Naik  
   - [Watch Video](https://www.youtube.com/watch?v=JxgmHe2NyeY)  
   - Views: 1,950,362  
   - Duration: 238 minutes and 7 seconds

2. **Title**: Machine Learning | What Is Machine Learning? | Introduction To Machine Learning | 2026 | Simplilearn  
   - [Watch Video](https://www.youtube.com/watch?v=ukzFI9rgwfU)  
   - Views: 5,443,196  
   - Duration: 47 minutes and 2 seconds

3. **Title**: AI, Machine Learning, Deep Learning and Generative AI Explained  
   - [Watch Video](https://www.youtube.com/watch?v=qYNweeDHiyU)  
   - Views: 3,165,839  
   - Duration: 60 minutes

4. **Title**: Most Machine Learning Courses Won't Get You Hired in 2026  
   - [Watch Video](https://www.youtube.com/watch?v=UZ_rK9gzVSc)  
   - Views: 5,563  
   - Duration: 6 minutes and 9 seconds

5. **Title**: Machine Learning for Everybody – Full Course  
   -

Building the Agent Loop

In [44]:
def run_agent(user_message: str):
    print(f"\nUser: {user_message}")
    print("-" * 50)
    
    # Start conversation with just the user message
    messages = [{"role": "user", "content": user_message}]
    
    # Keep looping until the model stops calling tools
    while True:
        response = ollama.chat(
            model="qwen2.5:3b",
            messages=messages,
            tools=list(tool_registry.values()),
        )
        
        # Add model's response to conversation history
        messages.append(response.message)
        
        # Check if the model wants to call a tool
        if response.message.tool_calls:
            # Handle every tool call the model requested
            for tool_call in response.message.tool_calls:
                function_name = tool_call.function.name
                function_args = tool_call.function.arguments
                
                print(f"Calling tool: {function_name}")
                print(f"Arguments: {function_args}")
                
                # Execute the tool
                function_to_call = tool_registry[function_name]
                tool_result = function_to_call(**function_args)
                
                print(f"Result: {str(tool_result)[:200]}...")
                print("-" * 50)
                
                # Add tool result to conversation history
                messages.append({
                    "role": "tool",
                    "tool_name": function_name,
                    "content": str(tool_result),
                })
        else:
            # No more tool calls — model has final answer
            print(f"\nFinal Answer:\n{response.message.content}")
            break




Testing the Agent:

In [45]:
run_agent("What are 3 trending videos right now?")


User: What are 3 trending videos right now?
--------------------------------------------------
Calling tool: get_trending_videos
Arguments: {'max_results': 3}
Result: [{'title': 'ASMR Live 🔴 #live #ASMR #satisfying #shorts #shortslive #shortsfeed #trending #coin 2026-05-27 12:25', 'video_id': 'KBGb55ZqMmY', 'url': 'https://www.youtube.com/watch?v=KBGb55ZqMmY', 'vie...
--------------------------------------------------

Final Answer:
Here are three trending videos on YouTube:

1. **Title:** ASMR Live 🔴 #live #ASMR #satisfying #shorts #shortslive #shortsfeed #trending #coin  
   - [YouTube Link](https://www.youtube.com/watch?v=KBGb55ZqMmY)
   - *Views:* 7,903

2. **Title:** Test Your Brain⚡ #YouTubeLive #LiveNow #Trending #viral  
   - [YouTube Link](https://www.youtube.com/watch?v=PbjcOBB2jOE)
   - *Views:* 5,670

3. **Title:** Ak 111k Live #ak111k #freefire #gamer #trending #ff #youtube #game #viral #pcgaming  
   - [YouTube Link](https://www.youtube.com/watch?v=A3OGF8AgSZs)
   - *Vie

In [46]:
run_agent("What is the title and view count of this video: https://www.youtube.com/watch?v=ukzFI9rgwfU")



User: What is the title and view count of this video: https://www.youtube.com/watch?v=ukzFI9rgwfU
--------------------------------------------------
Calling tool: extract_video_id
Arguments: {'url': 'https://www.youtube.com/watch?v=ukzFI9rgwfU'}
Result: ukzFI9rgwfU...
--------------------------------------------------
Calling tool: get_video_metadata
Arguments: {'video_id': 'ukzFI9rgwfU'}
Result: {'title': 'Machine Learning | What Is Machine Learning? | Introduction To Machine Learning | 2026 | Simplilearn', 'author': 'Simplilearn', 'views': 5443196, 'duration_seconds': 472, 'description': '"️...
--------------------------------------------------

Final Answer:
The title of the video is "Machine Learning | What Is Machine Learning? | Introduction To Machine Learning | 2026 | Simplilearn" and it has been viewed by 5,443,196 people. The duration of this video is approximately 472 seconds (about 7 minutes and 52 seconds). Additionally, here's some more context about the video: 
- Author: